In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU — go to Runtime > Change runtime type > T4 GPU")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics', 'supervision', 'scikit-learn', 'pandas',
    'opencv-python-headless'], check=True)
print("Dependencies installed")

Dependencies installed


In [ ]:
import os, sys

# Clonar o actualizar el repo (repo publico, no necesita token)
if not os.path.exists('/content/foostats_ai'):
    !git clone https://github.com/pipachiesa/foostats-ai.git /content/foostats_ai
else:
    !git -C /content/foostats_ai pull

os.chdir('/content/foostats_ai')
if '/content/foostats_ai' not in sys.path:
    sys.path.insert(0, '/content/foostats_ai')

# Copiar binarios grandes desde Drive (se quedan ahi permanentemente)
os.makedirs('models', exist_ok=True)
os.makedirs('input_videos', exist_ok=True)
!cp "/content/drive/MyDrive/foostats_ai/models/best.pt" models/
!cp "/content/drive/MyDrive/foostats_ai/input_videos/08fd33_4.mp4" input_videos/
print("Setup completo")


In [6]:
required = [
    'main.py',
    'models/best.pt',
    'input_videos/08fd33_4.mp4',
    'trackers/tracker.py',
    'teams_assigner/team_assigner.py',
    'camara_movement_estimator/camara_movement_estimator.py',
    'player_ball_assigner/player_ball_assigner.py',
    'speed_and_distance_estimator/speed_and_distance_estimator.py',
    'view_transformer/view_transformer.py',
    'utils/video_utils.py',
]

all_ok = True
for f in required:
    ok = os.path.exists(f)
    print(f"{'OK' if ok else 'MISSING'}  {f}")
    if not ok:
        all_ok = False

print("\nAll files present — ready to run." if all_ok else "\nFix missing files before running cell 6.")

OK  main.py
OK  models/best.pt
OK  input_videos/08fd33_4.mp4
OK  trackers/tracker.py
OK  teams_assigner/team_assigner.py
OK  camara_movement_estimator/camara_movement_estimator.py
OK  player_ball_assigner/player_ball_assigner.py
OK  speed_and_distance_estimator/speed_and_distance_estimator.py
OK  view_transformer/view_transformer.py
OK  utils/video_utils.py

All files present — ready to run.


In [ ]:
import os, time, importlib
from datetime import datetime

# Limpiar cache de modulos del proyecto
mods_to_clear = [k for k in __import__("sys").modules if any(k.startswith(p) for p in [
    "utils", "trackers", "teams_assigner", "camara_movement_estimator",
    "player_ball_assigner", "speed_and_distance_estimator",
    "view_transformer", "main"
])]
for mod in mods_to_clear:
    del __import__("sys").modules[mod]

os.makedirs("output_videos", exist_ok=True)
os.makedirs("stubs", exist_ok=True)

# Nombre unico por timestamp — nunca pisa outputs anteriores
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

import main as m
importlib.reload(m)
m.OUTPUT_PATH = f"output_videos/output_{timestamp}.avi"
print(f"Guardando como: {m.OUTPUT_PATH}")

start = time.time()
m.main()
elapsed = time.time() - start
print(f"
Done in {elapsed:.1f}s ({elapsed/60:.1f} min)")


In [ ]:
output = m.OUTPUT_PATH
if os.path.exists(output):
    print(f"OK: {output} ({os.path.getsize(output)/1e6:.1f} MB)")
else:
    print("No output — revisa errores arriba.")


In [21]:
import cv2
cap = cv2.VideoCapture('input_videos/08fd33_4.mp4')
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

ms_per_frame = 20  # T4 GPU estimate
est_min = (int(90 * 60 * fps) * ms_per_frame / 1000) / 60
print(f"Clip: {total} frames ({total/fps/60:.1f} min @ {fps:.0f}fps)")
print(f"Estimated for full 90-min match: ~{est_min:.0f} min on T4")

Clip: 750 frames (0.5 min @ 25fps)
Estimated for full 90-min match: ~45 min on T4
